What is CrewAI?

CrewAI is an agent-based orchestration framework designed to let multiple AI agents work together like a real-world team. Instead of a single large language model doing everything, CrewAI organizes roles, responsibilities, and workflows—similar to how humans collaborate in workplaces.

Think of it as building a team of AI specialists, where each agent has:

A role (what they are)

A goal (what they must accomplish)

A backstory (context that influences behavior)

A specific task (work assigned to them)

Optionally, tools and memory

# Agents to Research and Write an Article

Foundational concepts of multi-agent systems and get an overview of the crewAI framework.

For running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29


  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
  Using cached embedchain-0.1.128-py3-none-any.whl.metadata (9.2 kB)
  Using cached embedchain-0.1.127-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.126-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.125-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.124-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.123-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.122-py3-none-any.whl.metadata (9.3 kB)
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
  Using cached embedchain-0.1.121-py3-none-any.whl.metadata (9.3 kB)
  Using cached embedchain-0.1.120-py3-none-any.whl.metadata (9.3 kB)
  Using cach

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.0.2 which is incompatible.
langchain-experimental 0.0.65 requires langchain-community<0.3.0,>=0.2.16, but you have langchain-community 0.0.29 which is incompatible.
langchain-experimental 0.0.65 requires langchain-core<0.3.0,>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 0.1.53 which is incompatible.
langgraph-checkpoint 4.0.3 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [3]:
from crewai import Agent, Task, Crew

In [4]:
import os
from langchain_openai import ChatOpenAI
import utils

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner


In [5]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
	verbose=True
)

### Agent: Writer

In [6]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

### Agent: Editor

In [7]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [8]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [9]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [10]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 1 or 2 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [11]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see here.

In [12]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer  
Final Answer: 

**Content Plan Document for Blog Article on Artificial Intelligence**

---

### 1. Latest Trends, Key Players, and Noteworthy News in Artificial Intelligence

**Trends:**
- **Generative AI:** The rise of AI models that can create text, images, and music, such as OpenAI's ChatGPT and DALL-E.
- **AI Ethics and Regulation:** Increasing focus on ethical AI practices and the development of regulations to govern AI use.
- **AI in Healthcare:** Advancements in AI applications for diagnostics, personalized medicine

- Display the results of your execution as markdown in the notebook.

In [13]:
from IPython.display import Markdown
Markdown(result)

```markdown
# Understanding Artificial Intelligence: Trends, Impacts, and Future Directions

## I. Introduction

Artificial Intelligence (AI) has rapidly evolved from a niche area of research to a transformative force across various sectors. Its significance in today's world cannot be overstated, as AI technologies are reshaping industries, enhancing productivity, and influencing our daily lives. As we navigate this technological revolution, it is crucial to stay informed about the latest trends and developments in AI, which promise to enhance our capabilities while raising important ethical and societal questions.

In this article, we will explore the latest trends in AI, identify key players in the field, and discuss noteworthy news and developments. By understanding these elements, we can better appreciate the implications of AI and its potential to shape our future.

## II. Latest Trends in Artificial Intelligence

### Generative AI and Its Applications

One of the most exciting trends in AI is the rise of generative AI, which refers to models capable of creating text, images, and even music. Notable examples include OpenAI's ChatGPT and DALL-E, which have garnered significant attention for their ability to generate human-like text and stunning visuals. This technology is revolutionizing content creation and opening new avenues for creativity and innovation across various fields.

Generative AI has applications in marketing, entertainment, and education, allowing businesses to produce personalized content at scale. However, as these technologies advance, they also raise questions about authenticity and the potential for misuse, highlighting the need for ethical guidelines and regulations.

### The Role of AI in Healthcare

AI's impact on healthcare is another area of significant growth. From diagnostics to personalized medicine, AI technologies are enhancing patient care and operational efficiency. For instance, AI algorithms can analyze medical images with remarkable accuracy, assisting healthcare professionals in diagnosing conditions earlier and more reliably.

Moreover, AI-driven tools are being developed to tailor treatment plans to individual patients, taking into account their unique genetic makeup and health history. This shift towards personalized medicine not only improves patient outcomes but also optimizes resource allocation within healthcare systems. However, the integration of AI in healthcare necessitates careful consideration of data privacy and ethical implications.

### Automation and Its Impact on the Workforce

The automation of tasks through AI is transforming the workforce landscape. While automation can lead to increased efficiency and cost savings for businesses, it also raises concerns about job displacement. Many industries are experiencing a shift towards automated processes, resulting in significant changes to job roles and employment opportunities.

As AI continues to evolve, it is essential for businesses and policymakers to address these challenges proactively. Upskilling and reskilling initiatives will be crucial in preparing the workforce for the future, ensuring that individuals can adapt to the changing job market and leverage AI technologies effectively.

## III. Key Players in the AI Landscape

### Overview of Leading Companies and Their Contributions

The AI landscape is populated by several key players who are driving innovation and shaping the future of technology. OpenAI, known for its groundbreaking generative models, is at the forefront of AI research and development. Google DeepMind is another significant contributor, pioneering advancements in machine learning and AI applications.

IBM Watson focuses on AI solutions for business and healthcare, providing tools that enhance decision-making and operational efficiency. Microsoft is also integrating AI into its products and services, particularly through its Azure AI platform, which offers businesses scalable AI solutions. Additionally, NVIDIA is leading the charge in AI hardware and software, providing the necessary infrastructure for AI applications.

### Emerging Startups to Watch

In addition to established companies, numerous startups are emerging in the AI space, bringing fresh ideas and innovative solutions. These startups often focus on niche applications of AI, from enhancing customer service through chatbots to developing AI-driven analytics tools for businesses. Keeping an eye on these emerging players can provide valuable insights into the future direction of AI technology.

## IV. Noteworthy News and Developments

### Recent Breakthroughs and Innovations

The AI field is characterized by rapid advancements, with recent breakthroughs capturing the attention of both the tech community and the general public. The release of GPT-4, for instance, has showcased significant improvements in natural language processing, enabling more sophisticated interactions between humans and machines. Such innovations not only enhance user experience but also expand the potential applications of AI across various sectors.

### Legislative Changes and Ethical Discussions

As AI technologies continue to evolve, so too does the conversation around ethics and regulation. Recent legislative developments in various countries aim to establish frameworks for the responsible use of AI, addressing concerns related to data privacy, bias, and accountability. These discussions are crucial in ensuring that AI technologies are developed and deployed in ways that benefit society while minimizing potential harms.

## V. Understanding the Implications of AI

### Benefits of AI in Various Sectors

The benefits of AI are manifold, spanning across sectors such as healthcare, finance, and education. In healthcare, AI enhances diagnostic accuracy and personalizes treatment plans. In finance, AI algorithms can analyze market trends and assist in risk management. In education, AI-driven tools can provide personalized learning experiences, catering to the unique needs of each student.

These advancements not only improve efficiency but also foster innovation, enabling organizations to deliver better services and products to their customers.

### Addressing Concerns About Job Displacement and Ethics

Despite the numerous benefits, concerns about job displacement and ethical implications persist. As automation becomes more prevalent, it is essential to address the potential impact on employment. Policymakers and business leaders must work together to create strategies that support workforce transition, ensuring that individuals are equipped with the skills needed for the jobs of the future.

Moreover, ethical considerations surrounding AI, such as bias in algorithms and data privacy, must be prioritized. Establishing clear guidelines and fostering transparency in AI development will be crucial in building public trust and ensuring that AI serves the greater good.

## VI. Conclusion

In conclusion, understanding artificial intelligence is vital in today's rapidly changing technological landscape. By staying informed about the latest trends, key players, and ethical considerations, we can better navigate the complexities of AI and its implications for our lives and industries. As we look to the future, it is essential to embrace the opportunities that AI presents while remaining vigilant about its challenges.

## VII. Call to Action

We invite you to subscribe for updates on the latest AI trends and developments. Share this article on social media to help spread awareness about the importance of understanding artificial intelligence and its impact on our world.
```

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)